# 01 – Data Extraction

Pulls raw tract-level data for California from two public APIs and saves it unmodified to `data/raw/`.

**Sources**
- U.S. Census Bureau ACS 5-year estimates (2019–2023), via the Census API. Income, poverty, rent burden, education, employment, housing, and age.
- CDC PLACES census tract data (2025 release), via the CDC Socrata API. Model-based estimates for 40 health measures, using mostly 2023 survey data.

**Notes**
- PLACES values are modeled estimates of crude prevalence, not directly measured health outcomes.
- Raw files are not cleaned here. Census missing-data codes (large negative numbers) and top-coded income values are handled in the EDA notebook.
- Requires a Census API key stored as `CENSUS_API_KEY` in a `.env` file at the project root.

**Outputs**
- `data/raw/acs5_2023_tract_CA.csv`
- `data/raw/places_tract_CA.csv`

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
if not CENSUS_API_KEY:
    raise ValueError("CENSUS_API_KEY not found. Check your .env file.")
SOCRATA_APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN")  # optional, only raises rate limits

# --- Parameters: change these here, nowhere else ---
ACS_YEAR = 2023              # ACS 5-year vintage (2019-2023)
STATE_FIPS = "06"            # California
STATE_ABBR = "CA"
PLACES_DATASET_ID = "cwsq-ngmh"   # CDC PLACES census tract dataset - ID confirmed on cdc.gov

# https://data.cdc.gov/500-Cities-Places/PLACES-Local-Data-for-Better-Health-Census-Tract-D/cwsq-ngmh/about_data

# Paths are relative to the notebooks folder, so ../ backs up to the project root
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# create a dictionary which maps each census variable code to a clear description
# the codes get requested from the census api

# Browse more at: https://api.census.gov/data/2023/acs/acs5/variables.html
ACS_VARS = {
    # Income / poverty
    "B19013_001E": "Median household income",
    "B17001_001E": "Poverty universe (total pop)",
    "B17001_002E": "Pop below poverty level",
    # Rent burden (GRAPI)
    "B25070_001E": "Renter households (GRAPI universe)",
    "B25070_002E": "Rent is less than 10% of income",
    "B25070_003E": "Rent 10.0-14.9% of income",
    "B25070_004E": "Rent 15.0-19.9% of income",
    "B25070_005E": "Rent 20.0-24.9% of income",
    "B25070_006E": "Rent 25.0-29.9% of income",
    "B25070_007E": "Rent 30.0-34.9% of income",
    "B25070_008E": "Rent 35.0-39.9% of income",
    "B25070_009E": "Rent 40.0-49.9% of income",
    "B25070_010E": "Rent 50%+ of income",
    # Education (adults 25+): codes 002-016 = less than a high school diploma
    "B15003_001E": "Adults 25+ (education universe)",
    **{f"B15003_{i:03d}E": f"Education attainment level {i}" for i in range(2, 17)},
    # Employment
    "B23025_003E": "Civilian labor force",
    "B23025_005E": "Unemployed",
    # Housing
    "B25002_001E": "Total housing units",
    "B25002_003E": "Vacant housing units",
    "B25003_001E": "Occupied units (tenure universe)",
    "B25003_003E": "Renter-occupied units",
    "B25064_001E": "Median gross rent",
    # Population
    "B01003_001E": "Total population",
    "B01002_001E": "Median age",
}
assert len(ACS_VARS) <= 49, "Census API allows 50 variables per call"

In [3]:
def fetch_acs(year, variables, state_fips, api_key):
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": ",".join(["NAME"] + list(variables)),
        "for": "tract:*",
        "in": f"state:{state_fips}",
        "key": api_key,
    }
    r = requests.get(url, params=params, timeout=60)
    if r.status_code != 200 or "json" not in r.headers.get("Content-Type", ""):
        raise RuntimeError(
            f"Census API problem. status={r.status_code}, "
            f"content-type={r.headers.get('Content-Type')}, "
            f"body={r.text[:300]!r}"
        )    
    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df["tract_fips"] = df["state"] + df["county"] + df["tract"]  # 11-digit string
    for col in variables:
        df[col] = pd.to_numeric(df[col], errors="coerce")  # sentinels are NOT removed here
    return df


def fetch_places(dataset_id, state_abbr, app_token=None, page_size=50000):
    """Pull every row for one state, paging through the Socrata API."""
    url = f"https://data.cdc.gov/resource/{dataset_id}.json"
    headers = {"X-App-Token": app_token} if app_token else {}
    rows, offset = [], 0
    while True:
        params = {
            "$where": f"stateabbr='{state_abbr}'",
            "$order": ":id",            # stable ordering so pages don't overlap
            "$limit": page_size,
            "$offset": offset,
        }
        r = requests.get(url, params=params, headers=headers, timeout=120)
        if r.status_code != 200 or "json" not in r.headers.get("Content-Type", ""):
            raise RuntimeError(
                f"PLACES API problem. status={r.status_code}, "
                f"content-type={r.headers.get('Content-Type')}, "
                f"body={r.text[:300]!r}"
            )        
        batch = r.json()
        rows.extend(batch)
        print(f"  fetched {len(rows):,} rows so far")
        if len(batch) < page_size:
            break
        offset += page_size
    # Dropping geolocation: a nested field (non-atomic) 
    # that doesn’t save cleanly to CSV, and we join on tract ID, not coordinates.    
    return pd.DataFrame(rows).drop(columns=["geolocation"], errors="ignore") 

In [4]:
print("Fetching ACS...")
acs = fetch_acs(ACS_YEAR, ACS_VARS.keys(), STATE_FIPS, CENSUS_API_KEY)

print("Fetching PLACES...")
places = fetch_places(PLACES_DATASET_ID, STATE_ABBR, SOCRATA_APP_TOKEN)

Fetching ACS...
Fetching PLACES...
  fetched 50,000 rows so far
  fetched 100,000 rows so far
  fetched 150,000 rows so far
  fetched 200,000 rows so far
  fetched 250,000 rows so far
  fetched 300,000 rows so far
  fetched 350,000 rows so far
  fetched 362,792 rows so far


In [5]:
print("ACS shape:", acs.shape)
print("ACS tract_fips lengths:", acs["tract_fips"].str.len().value_counts().to_dict())

# Census uses large negative numbers (e.g. -666666666) for "no data"
neg = (acs[list(ACS_VARS)] < 0).sum()
print("\nACS columns with negative sentinel values:\n", neg[neg > 0])

print("\nPLACES shape:", places.shape)
print("PLACES columns:", list(places.columns))
print("Survey year values:", places["year"].unique())
print("Value types:", places["datavaluetypeid"].unique())
print("Measures:", sorted(places["measureid"].unique()))

acs_tracts = set(acs["tract_fips"])
places_tracts = set(places["locationid"])
print(f"\nTracts in ACS: {len(acs_tracts):,} | in PLACES: {len(places_tracts):,} "
      f"| overlap: {len(acs_tracts & places_tracts):,}")

ACS shape: (9129, 43)
ACS tract_fips lengths: {11: 9129}

ACS columns with negative sentinel values:
 B19013_001E    126
B25064_001E    301
B01002_001E     51
dtype: int64

PLACES shape: (362792, 21)
PLACES columns: ['year', 'stateabbr', 'statedesc', 'countyname', 'countyfips', 'locationname', 'datasource', 'category', 'measure', 'data_value_unit', 'data_value_type', 'data_value', 'low_confidence_limit', 'high_confidence_limit', 'totalpopulation', 'totalpop18plus', 'locationid', 'categoryid', 'measureid', 'datavaluetypeid', 'short_question_text']
Survey year values: ['2023' '2022']
Value types: ['CrdPrv']
Measures: ['ACCESS2', 'ARTHRITIS', 'BINGE', 'BPHIGH', 'BPMED', 'CANCER', 'CASTHMA', 'CHD', 'CHECKUP', 'CHOLSCREEN', 'COGNITION', 'COLON_SCREEN', 'COPD', 'CSMOKING', 'DENTAL', 'DEPRESSION', 'DIABETES', 'DISABILITY', 'EMOTIONSPT', 'FOODINSECU', 'FOODSTAMP', 'GHLTH', 'HEARING', 'HIGHCHOL', 'HOUSINSECU', 'INDEPLIVE', 'LACKTRPT', 'LONELINESS', 'LPA', 'MAMMOUSE', 'MHLTH', 'MOBILITY', 'OBESI

In [6]:
acs.to_csv(RAW_DIR / f"acs5_{ACS_YEAR}_tract_{STATE_ABBR}.csv", index=False)
places.to_csv(RAW_DIR / f"places_tract_{STATE_ABBR}.csv", index=False)

# In the EDA notebook, load IDs as strings so leading zeros survive:
# pd.read_csv(path, dtype={"tract_fips": str})
# pd.read_csv(path, dtype={"locationid": str})